In [24]:
import pandas as pd
import numpy as np

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from catboost import CatBoostClassifier

In [26]:
test=pd.read_csv("C:/Users/Administrator/Desktop/채원이/test_data_fin.csv", encoding="utf-8")
train=pd.read_csv("C:/Users/Administrator/Desktop/채원이/train_data_fin.csv",  encoding="utf-8")

In [27]:
#dummy함수 정의
import pandas as pd

def preprocess_train_test(
    train: pd.DataFrame,
    test: pd.DataFrame,
    dummy_cols=("학년", "전임교원_확보율", "입학세부구분"),
    drop_cols=("혁신수업", "연구재단등재지", "Unnamed: 0","계열"),
    ):
    train = train.copy()
    test = test.copy()

    # 제외 변수 drop (train/test 둘 다에 없을 수도 있으니 errors="ignore")
    train = train.drop(columns=list(drop_cols), errors="ignore")
    test  = test.drop(columns=list(drop_cols), errors="ignore")

    # 더미 처리 대상 중 실제로 존재하는 컬럼만
    dummy_cols = [c for c in dummy_cols if (c in train.columns or c in test.columns)]

    # train+test를 합쳐서 한 번에 더미 만들고 다시 분리 (컬럼 불일치 방지)
    n_train = len(train)
    both = pd.concat([train, test], axis=0, ignore_index=True)

    # 더미 처리, 범주형 -> 숫자
    both = pd.get_dummies(both, columns=dummy_cols, drop_first=False)

    X_train = both.iloc[:n_train, :].reset_index(drop=True)
    X_test  = both.iloc[n_train:, :].reset_index(drop=True)

    return X_train, X_test

In [28]:
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    fbeta_score, f1_score, accuracy_score
)
from sklearn.base import clone

BETA = 2.0
N_SPLITS = 5
RANDOM_STATE = 42

# 타깃 변수명
target_col = "학적상태"

# 1) y 분리
y_train = train[target_col].copy()
y_test  = test[target_col].copy()

# 2) X 분리
X_train_raw = train.drop(columns=[target_col])
X_test_raw  = test.drop(columns=[target_col])   # ← 여기 수정

# 3) X 전처리
X_train, X_test = preprocess_train_test(
    X_train_raw,
    X_test_raw,
    dummy_cols=("학년", "전임교원_확보율", "입학세부구분"),
    drop_cols=("혁신수업", "연구재단등재지", "Unnamed: 0","계열"),
)


In [11]:
X_train

,일반휴학학기수,평점평균,취업률,평균취득학점이상,학년_1,학년_2,학년_3,학년_4,전임교원_확보율_-1,전임교원_확보율_0,전임교원_확보율_1,입학세부구분_0,입학세부구분_1,입학세부구분_2
0,0,1.49,71,0,True,False,False,False,True,False,False,False,False,True
1,0,2.44,71,1,True,False,False,False,True,False,False,False,True,False
2,3,4.00,71,0,True,False,False,False,True,False,False,False,True,False
3,4,3.73,71,0,True,False,False,False,True,False,False,False,True,False
4,0,3.92,71,1,True,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14243,0,3.74,71,0,False,True,False,False,True,False,False,True,False,False
14244,2,4.18,71,1,False,True,False,False,True,False,False,False,False,True
14245,0,3.51,71,1,False,True,False,False,True,False,False,False,True,False
14246,0,4.24,71,0,False,True,False,False,True,False,False,False,True,False


# Catboost

In [29]:
import sys
!"{sys.executable}" -m pip install catboost


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Users\Administrator\.conda\envs\py3env\python.exe -m pip install --upgrade pip


In [30]:
from catboost import CatBoostClassifier

def fit_catboost_predict_proba(X_tr, y_tr, X_va, y_va, params):
    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0,
        thread_count=-1
    )

    model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    proba = model.predict_proba(X_va)[:, 1]
    return model, proba


# SVM

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def make_svm_model(X_example, random_state=42):
    # 더미화가 끝났으니 전체 컬럼을 numeric으로 보고 스케일링만
    num_cols = X_example.columns.tolist()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop"
    )

    return Pipeline([
        ("prep", preprocess),
        ("clf", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=False,
            random_state=random_state
        ))
    ])

m = make_svm_model(X_train, random_state=RANDOM_STATE)
m.fit(X_train, y_train)

scores = m.decision_function(X_test)

print("score min / mean / max:", scores.min(), scores.mean(), scores.max())
print("positives predicted (score>=0):", (scores >= 0).sum(), "/", len(scores))
print("true positive rate:", y_test.mean())


score min / mean / max: -1.0534624807952493 -1.0076814218240295 -0.9586091494179478
positives predicted (score>=0): 0 / 6106
true positive rate: 0.020962987225679658


# RF

In [40]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

def make_rf_model(X_example, random_state=RANDOM_STATE):
    return Pipeline([
        ("clf", RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1
        ))
    ])


## gridsearch (train으로 -> valid)

In [41]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import roc_auc_score, average_precision_score

N_SPLITS = 5
RANDOM_STATE = 42

def custom_search_Xy(X, y, model_name, make_model_fn=None, param_grid=None):
    if not param_grid:
        raise ValueError("param_grid must be provided")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []

    for params_raw in ParameterGrid(param_grid):
        fold_roc, fold_pr = [], []

        for tr_idx, va_idx in skf.split(X, y):
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

            if model_name == "catboost":
                m, score = fit_catboost_predict_proba(X_tr, y_tr, X_va, y_va, params_raw)
            else:
                if make_model_fn is None:
                    raise ValueError("make_model_fn must be provided for non-catboost models")

                m = make_model_fn(X_tr)

                params_fit = dict(params_raw)      # fit용 복사본
                params_fit.pop("class_weights", None)
                params_fit.pop("scale_pos_weight", None)

                m.set_params(**params_fit)
                m.fit(X_tr, y_tr)

                # SVM(probability=False)면 decision_function이 score
                if hasattr(m, "predict_proba"):
                    score = m.predict_proba(X_va)[:, 1]
                else:
                    score = m.decision_function(X_va)

            fold_roc.append(roc_auc_score(y_va, score))
            fold_pr.append(average_precision_score(y_va, score))

        rows.append({
            "model": model_name,
            "params": dict(params_raw),  # ✅ 원본 그대로 저장
            "mean_ROC-AUC": float(np.mean(fold_roc)),
            "mean_PR-AUC": float(np.mean(fold_pr)),
        })

    res = (pd.DataFrame(rows)
           .sort_values(["mean_ROC-AUC", "mean_PR-AUC"], ascending=False)
           .reset_index(drop=True))
    best = res.iloc[0].to_dict()
    return best, res




In [42]:
svm_grid = {
    "clf__kernel": ["rbf"],
    "clf__C": [0.25, 1.0, 4.0, 16.0, 64.0],
    "clf__gamma": ["scale", 0.001, 0.01, 0.1, 1.0],
    "clf__class_weight": [None, "balanced"],   
}

rf_grid = {
    "clf__n_estimators": [300, 600],
    "clf__max_depth": [None, 10, 20, 40],
    "clf__min_samples_leaf": [1, 3, 5],
    "clf__max_features": ["sqrt", 0.5],
    "clf__class_weight": [None, "balanced"],   
}

cat_grid = {
    "iterations": [800, 1200],
    "learning_rate": [0.03, 0.05, 0.1],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1, 3, 10],
    "auto_class_weights": [None, "Balanced"], 
}

## best parameter search includes class_weight

In [43]:
best_svm, res_svm = custom_search_Xy(
    X_train, y_train,
    model_name="svm",
    make_model_fn=make_svm_model,
    param_grid=svm_grid
)

best_rf, res_rf = custom_search_Xy(
    X_train, y_train,
    model_name="rf",
    make_model_fn=make_rf_model,
    param_grid=rf_grid
)  

best_cat, res_cat = custom_search_Xy(
    X_train, y_train,
    model_name="catboost",
    param_grid=cat_grid
)

best_svm, best_rf, best_cat


({'model': 'svm',
  'params': {'clf__C': 0.25,
   'clf__class_weight': 'balanced',
   'clf__gamma': 0.001,
   'clf__kernel': 'rbf'},
  'mean_ROC-AUC': 0.898950990530657,
  'mean_PR-AUC': 0.1339505289497708},
 {'model': 'rf',
  'params': {'clf__class_weight': None,
   'clf__max_depth': 10,
   'clf__max_features': 0.5,
   'clf__min_samples_leaf': 3,
   'clf__n_estimators': 600},
  'mean_ROC-AUC': 0.9117447460904741,
  'mean_PR-AUC': 0.19114899001507452},
 {'model': 'catboost',
  'params': {'auto_class_weights': None,
   'depth': 6,
   'iterations': 800,
   'l2_leaf_reg': 1,
   'learning_rate': 0.05},
  'mean_ROC-AUC': 0.9206428024592199,
  'mean_PR-AUC': 0.21574516010480527})

## Best params 지표 (Original data)

In [46]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score, precision_score

def metrics_simple(y_true, score, thr):
    pred = (score >= thr).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, score),
        "Recall": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Pred_Pos": int(pred.sum()),
        "Thr_used": thr,
    }

target_col = "학적상태"
y_train = train[target_col].copy()
y_test  = test[target_col].copy()

def fit_final_and_eval(model_name, best):
    if model_name == "catboost":
        m = CatBoostClassifier(
            **best["params"],
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=0,
            thread_count=-1
        )
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5

    elif model_name == "svm":
        m = make_svm_model(X_train)
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.decision_function(X_test)
        thr = 0.0

    elif model_name == "rf":
        m = make_rf_model(X_train)     
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5

    mets = metrics_simple(y_test, score, thr)
    return {"model": model_name, **mets}


summary


,model,ROC-AUC,Recall,F1,Accuracy,Precision,Pred_Pos,Thr_used
0,svm,0.887168,0.945312,0.125911,0.724861,0.067447,1794,0.0
1,catboost,0.902058,0.015625,0.029630,0.978546,0.285714,7,0.5
2,rf,0.894146,0.007812,0.015385,0.979037,0.500000,2,0.5


In [47]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# =========================
# 0) 공통 유틸
# =========================
def _to_pred(score, thr):
    return (score >= thr).astype(int)

def print_confmat(name, y_true, pred):
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    print(f"\n===== {name} =====")
    print("Confusion matrix [ [TN FP], [FN TP] ]:")
    print(cm)
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print("Pred positives:", int(pred.sum()), "/", len(pred))
    # 분류리포트(precision/recall/f1)도 같이
    print("\nClassification report:")
    print(classification_report(y_true, pred, digits=4, zero_division=0))

# =========================
# 1) 모델별 score/proba + 혼동행렬
# =========================
def get_model_score_and_thr(model_name, best):
    """
    반환:
      m     : fitted model
      score : (n_samples,)  -- SVM이면 decision_function, 나머지는 proba[:,1]
      thr   : 기본 임계값 (SVM=0.0, proba=0.5)
    """
    if model_name == "svm":
        m = make_svm_model(X_train, random_state=RANDOM_STATE)
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.decision_function(X_test)
        thr = 0.0
        return m, score, thr

    elif model_name == "rf":
        m = make_rf_model(X_train)
        m.set_params(**best["params"])
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5
        return m, score, thr

    elif model_name == "catboost":
        m = CatBoostClassifier(
            **best["params"],
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=RANDOM_STATE,
            verbose=0,
            thread_count=-1
        )
        m.fit(X_train, y_train)
        score = m.predict_proba(X_test)[:, 1]
        thr = 0.5
        return m, score, thr

    else:
        raise ValueError("Unknown model_name")

# 1-1) 세 모델 혼동행렬 출력
models = [
    ("SVM", best_svm),
    ("RF", best_rf),
    ("CatBoost", best_cat),
]

fitted = {}  # 나중 탐색에 재사용
for name, best in models:
    m, score, thr = get_model_score_and_thr(name.lower() if name!="CatBoost" else "catboost", best)
    pred = _to_pred(score, thr)
    print_confmat(name, y_test, pred)
    fitted[name] = (m, score, thr)

# =========================
# 2) recall 낮은 원인 탐색 (원인 확정용)
# =========================

def explore_recall_issue(name, y_true, score, thr):
    print(f"\n\n######## Recall diagnostic: {name} ########")

    # (A) score 분포: 양성/음성 평균, 분위수
    pos = score[y_true == 1]
    neg = score[y_true == 0]

    def q(x):
        return np.quantile(x, [0.0, 0.1, 0.5, 0.9, 1.0])

    print("\n[A] Score distribution by class")
    print("  pos count:", len(pos), "neg count:", len(neg))
    if len(pos) > 0:
        print("  pos mean:", float(np.mean(pos)), "quantiles:", q(pos))
    print("  neg mean:", float(np.mean(neg)), "quantiles:", q(neg))
    print("  thr used:", thr)

    # (B) 임계값에 따른 recall/precision 변화 (짧게)
    #  - SVM(score): 기본 thr=0이지만, recall이 0이면 "thr 근처"를 훑어봄
    #  - proba: 0.5 중심으로 훑어봄
    if name == "SVM":
        # SVM은 score 범위가 좁을 수 있어: score 분위수 기반으로 threshold 후보 생성
        cand = np.unique(np.quantile(score, np.linspace(0.0, 1.0, 21)))
    else:
        cand = np.linspace(0.0, 1.0, 21)

    rows = []
    for t in cand:
        pred = (score >= t).astype(int)
        tp = np.sum((pred == 1) & (y_true == 1))
        fp = np.sum((pred == 1) & (y_true == 0))
        fn = np.sum((pred == 0) & (y_true == 1))

        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        pred_pos = int(pred.sum())

        rows.append((float(t), recall, precision, pred_pos))

    df_curve = pd.DataFrame(rows, columns=["thr", "recall", "precision", "pred_pos"])
    # recall 상위 몇 개만 보기(너무 길어지니까)
    print("\n[B] Threshold sweep (top by recall, then precision)")
    display(df_curve.sort_values(["recall", "precision"], ascending=False).head(10))

    # (C) “양성 점수가 임계값을 넘길 수 있는지” 체크
    if len(pos) > 0:
        print("\n[C] Feasibility check")
        print("  max(pos score) =", float(np.max(pos)))
        print("  min(neg score) =", float(np.min(neg)))
        if np.max(pos) < thr:
            print("  -> 양성 score의 최대가 thr보다 작음: 현재 thr로는 recall이 구조적으로 낮을 수밖에 없음.")
        else:
            print("  -> 양성 score가 thr를 넘는 구간은 존재함: recall 0이면 다른 원인(학습/분포/모델)이 의심됨.")

# 2-1) 모델별 진단 실행
# (SVM은 score가 decision_function이므로 name에 맞춰 넣어줌)
explore_recall_issue("SVM", y_test, fitted["SVM"][1], fitted["SVM"][2])
explore_recall_issue("RF", y_test, fitted["RF"][1], fitted["RF"][2])
explore_recall_issue("CatBoost", y_test, fitted["CatBoost"][1], fitted["CatBoost"][2])



===== SVM =====
Confusion matrix [ [TN FP], [FN TP] ]:
[[4305 1673]
 [   7  121]]
TN=4305, FP=1673, FN=7, TP=121
Pred positives: 1794 / 6106

Classification report:
              precision    recall  f1-score   support

           0     0.9984    0.7201    0.8367      5978
           1     0.0674    0.9453    0.1259       128

    accuracy                         0.7249      6106
   macro avg     0.5329    0.8327    0.4813      6106
weighted avg     0.9789    0.7249    0.8218      6106


===== RF =====
Confusion matrix [ [TN FP], [FN TP] ]:
[[5977    1]
 [ 127    1]]
TN=5977, FP=1, FN=127, TP=1
Pred positives: 2 / 6106

Classification report:
              precision    recall  f1-score   support

           0     0.9792    0.9998    0.9894      5978
           1     0.5000    0.0078    0.0154       128

    accuracy                         0.9790      6106
   macro avg     0.7396    0.5038    0.5024      6106
weighted avg     0.9691    0.9790    0.9690      6106


===== CatBoost =====

,thr,recall,precision,pred_pos
5,-1.003741,1.000000,0.027954,4579
4,-1.005169,1.000000,0.026203,4885
3,-1.006994,1.000000,0.024663,5190
2,-2.943902,1.000000,0.023290,5496
1,-2.977246,1.000000,0.022069,5800
0,-2.987763,1.000000,0.020963,6106
6,-1.002505,0.992188,0.029715,4274
10,-0.999321,0.984375,0.041271,3053
9,-1.000103,0.984375,0.037522,3358
8,-1.000806,0.984375,0.034389,3664



[C] Feasibility check
  max(pos score) = 1.0047150015368325
  min(neg score) = -2.98776315168269
  -> 양성 score가 thr를 넘는 구간은 존재함: recall 0이면 다른 원인(학습/분포/모델)이 의심됨.


######## Recall diagnostic: RF ########

[A] Score distribution by class
  pos count: 128 neg count: 5978
  pos mean: 0.1263892256203077 quantiles: [0.         0.01630873 0.07437862 0.32677921 0.53600284]
  neg mean: 0.017013102918072797 quantiles: [0.00000000e+00 0.00000000e+00 9.65848648e-05 5.24877195e-02
 5.41895956e-01]
  thr used: 0.5

[B] Threshold sweep (top by recall, then precision)


,thr,recall,precision,pred_pos
0,0.00,1.000000,0.020963,6106
1,0.05,0.656250,0.118143,711
2,0.10,0.429688,0.182724,301
3,0.15,0.328125,0.230769,182
4,0.20,0.234375,0.283019,106
5,0.25,0.179688,0.348485,66
6,0.30,0.117188,0.357143,42
7,0.35,0.054688,0.304348,23
8,0.40,0.046875,0.400000,15
9,0.45,0.023438,0.500000,6



[C] Feasibility check
  max(pos score) = 0.5360028368097891
  min(neg score) = 0.0
  -> 양성 score가 thr를 넘는 구간은 존재함: recall 0이면 다른 원인(학습/분포/모델)이 의심됨.


######## Recall diagnostic: CatBoost ########

[A] Score distribution by class
  pos count: 128 neg count: 5978
  pos mean: 0.11643038569790844 quantiles: [2.09089482e-06 2.43264541e-03 4.45419923e-02 3.48247671e-01
 8.10146915e-01]
  neg mean: 0.014392964275966324 quantiles: [9.38041353e-08 1.11070837e-06 1.91813448e-05 3.21695244e-02
 9.10115274e-01]
  thr used: 0.5

[B] Threshold sweep (top by recall, then precision)


,thr,recall,precision,pred_pos
0,0.00,1.000000,0.020963,6106
1,0.05,0.476562,0.129237,472
2,0.10,0.335938,0.167315,257
3,0.15,0.265625,0.203593,167
4,0.20,0.210938,0.234783,115
5,0.25,0.140625,0.230769,78
6,0.30,0.132812,0.253731,67
7,0.35,0.101562,0.270833,48
8,0.40,0.070312,0.243243,37
10,0.50,0.039062,0.217391,23



[C] Feasibility check
  max(pos score) = 0.8101469148009036
  min(neg score) = 9.380413534621873e-08
  -> 양성 score가 thr를 넘는 구간은 존재함: recall 0이면 다른 원인(학습/분포/모델)이 의심됨.
